# Modelo Basado en Procesos
## La propuesta: Biogeochemical Flux Model 0D

**Objetivo**

**Ventajas** Un modelo basado en procesos respeta las leyes biológicas y físicas; la calibración corrige parámetros y la asimilación el estado con datos reales.

**Desventajas** Requiere calibrar muchos parámetros biológicos difíciles de medir.

**pyfabm** es el *python driver for the Framework for Aquatic Biogeochemical Models (FABM)*. Es una especie de wrapper del código fortran compilado que expone su API a python. El código fortran de BFM está en `seamless-notebooks/extern/ogs/`.
Así que después de compilar con `bash ./my_install`, pyfabm está disponible en el entorno y se carga con `import pyfabm`.

In [ ]:
import os,sys
import numpy as np
import scipy.integrate
import pyfabm
import netCDF4 as nc
from scipy.fft import fft, fftfreq
from scipy.signal import find_peaks
import math
import subprocess
#import statsmodels.api as sm
from scipy.stats import variation

**fabm.yaml** es el archivo básico de configuración de un modelo biogeoquímico, define las variables de estado que se utilizan, sus parámetros y el acoplamiento entre ellas. Debe colocarse en el directorio del setup de la estación E2GI, en `seamless-notebooks/setups/E2GI/`.
pyfabm lee fabm.yaml, y crea el objeto `model` que es un objeto python que actúa como interfaz hacia un modelo FABM que sigue ejecutándose en código compilado.
Tengo una versión simplificada de BFM traducida a R, pero lo mantengo como plan B.

In [ ]:
# Create model (loads fabm.yaml)
model = pyfabm.Model('fabm.yaml')

In [ ]:
allvariables = list(model.state_variables) + list(model.dependencies)
#list(model.diagnostic_variables)

In [ ]:
# Configure the environment
# Note: the set of environmental dependencies depends on the loaded biogeochemical model.
model.dependencies['cell_thickness'].value = 1.
model.dependencies['temperature'].value = 15.
model.dependencies['practical_salinity'].value = 30.
model.dependencies['density'].value = 1000.
model.dependencies['depth'].value = 1.
model.dependencies['pressure'].value = 1.
#model.dependencies['isBen'].value = 1.
model.dependencies['longitude'].value = 0.
model.dependencies['latitude'].value = 0.
model.dependencies['surface_downwelling_shortwave_flux'].value = 50.
model.dependencies['surface_air_pressure'].value = 1.
model.dependencies['wind_speed'].value = 5.
model.dependencies['mole_fraction_of_carbon_dioxide_in_air'].value = 390.
model.dependencies['number_of_days_since_start_of_the_year'].value = 1.

In [ ]:
# Spectral light dependencies
model.dependencies['surf_direct_downward_irradiance_0250_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0325_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0350_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0375_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0400_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0425_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0450_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0475_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0500_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0525_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0550_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0575_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0600_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0625_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0650_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0675_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0700_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0725_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0775_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0850_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_0950_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1050_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1150_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1250_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1350_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1450_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1550_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1650_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1750_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_1900_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_2200_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_2900_nm'].value = 5.
model.dependencies['surf_direct_downward_irradiance_3700_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0250_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0325_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0350_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0375_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0400_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0425_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0450_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0475_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0500_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0525_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0550_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0575_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0600_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0625_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0650_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0675_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0700_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0725_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0775_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0850_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_0950_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1050_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1150_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1250_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1350_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1450_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1550_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1650_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1750_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_1900_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_2200_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_2900_nm'].value = 5.
model.dependencies['surf_diffuse_downward_irradiance_3700_nm'].value = 5.

In [ ]:
# Verify the model is ready to be used
model.cell_thickness=1.
assert model.checkReady()

In [ ]:
#ejemplo=model.diagnostic_variables['P1_11/run'].value
#ejemplo.item()
#model.diagnostic_variables['P1_11/run'].index

## Enumerating state variables
#for variable in model.state_variables:
#   print(f"  {variable.name} = {variable.long_name} ({variable.units})")

## Enumerating diagnostic variables
#for variable in model.diagnostic_variables:
#   print(f"  {variable.name} = {variable.long_name} ({variable.units})")

In [ ]:
# Time derivative
def dy(t0, y):
    model.state[:] = y
    return model.getRates()

In [ ]:
# Time-integrate over 1000 days (note: FABM's internal time unit is seconds!)
t_eval = np.linspace(0, 730.*86400, 100000)
deltaT=t_eval[1]-t_eval[0]
#t_eval = np.linspace(0, 3650.*86400, 300000)
sol = scipy.integrate.solve_ivp(dy, [0., 730.*86400], model.state, t_eval=t_eval) # needs at least two years to converge, returns t and y
#sol = scipy.integrate.odeint(dy, model.state, t_eval) # returns only y
#sol = scipy.integrate.OdeSolver(dy, 0., model.state, 365.*86400, False) # returns t (only initial) and y

t = sol.t/86400   # timesteps in days
y = sol.y.T       # sol.y transposed
Nt=t.shape[0]
deltaT=t[1]-t[0]
laststeps = int(Nt/10) #compute the indicators just for this last timesteps
freq = (1/deltaT) * np.linspace(0,laststeps/2,int(laststeps/2)) / laststeps

In [ ]:
# Guido solution
## Time-integrate over 1000 days (note: FABM's internal time unit is seconds!)
#t_eval = np.linspace(0, 730.*86400, 100000)
#dt = (t_eval[-1]-t_eval[0])/len(t_eval)
#y = np.zeros((len(t_eval),386))
#eT = np.zeros(len(t_eval))

##initial value
#y[0,:]=model.state[:]

#for i in range(len(t_eval)):
#          dy = model.getRates()
#          for j in range(len(model.state)):
#              if i!=0: #do not redo initial time
#                y[i,j]=y[i-1,j]+dy[j]*dt
#          eT[i] = model.diagnostic_variables['P1_11/run'].value
#          model.state[:]=y[i,:] #update model state
#print(eT)
#t = t_eval/86400 # in days

# Plot results
#import pylab
#pylab.plot(t, y)
#pylab.legend([variable.path for variable in model.state_variables])
#pylab.show()

In [ ]:
# Update model.state
model.state[:]=y[79244,:]

In [ ]:
# Save results
fileoutput = 'result.nc'
f = nc.Dataset(fileoutput, mode='w')

lat_dim = f.createDimension('lat', 1)
lon_dim = f.createDimension('lon', 1)
dep_dim = f.createDimension('z', 1)
time_dim = f.createDimension('time', Nt)

lat = f.createVariable('lat', np.float32, ('lat',))
lat.units = 'degrees_north'
lat.long_name = 'latitude'
f.variables['lat'][:]=0

lon = f.createVariable('lon', np.float32, ('lon',))
lon.units = 'degrees_east'
lon.long_name = 'longitude'
f.variables['lon'][:]=0

time = f.createVariable('time', np.float64, ('time',))
time.units = 'days'
time.long_name = 'time'
f.variables['time'][:]=t

depth = f.createVariable('z', np.float32, ('z',))
depth.units = 'meters'
depth.long_name = 'depth'
f.variables['z'][:]=1

for v,variable in enumerate(model.state_variables):
   ncvar = variable.name.replace("/","_")
   var = f.createVariable(ncvar, np.float64, ('time', 'z','lat','lon'))
   var.units = variable.units
   var.long_name = variable.long_name
   f.variables[ncvar][:]=y[:,v]

## Enumerating diagnostic variables
#for variable in model.diagnostic_variables:
#   print(f"  {variable.name} = {variable.long_name} ({variable.units})")
list1=['test']
substrings = ['_et','_iN','_tN','_eiPPY','_sum','_rugc','_run','_rrc','_rum']
for v,variable in enumerate(model.diagnostic_variables):
   ncvar = variable.name.replace("/","_")
   if any(substring in ncvar for substring in substrings):
       if model.diagnostic_variables[variable.name].value is None:
         print(" output is None! ")
       else:
         print(f"  {variable.name} = {variable.long_name} ({variable.units})")
         list1.append(variable.name)

diag_dim = f.createDimension('diag', len(list1))
diag_value = f.createVariable('diag_value', np.float64, ('diag',))
diag_name = f.createVariable('diag_name', 'S20', ('diag',))
diag_value.units = '-'
diag_value.long_name = 'value diagnostics steady state'
diag_name.units = '-'
diag_name.long_name = 'name diagnostics steady state'

n=0
list2=[1]
substrings = ['_et','_iN','_tN','_eiPPY','_sum','_rugc','_run','_rrc','_rum']
for v,variable in enumerate(model.diagnostic_variables):
   ncvar = variable.name.replace("/","_")
   if any(substring in ncvar for substring in substrings):
       if model.diagnostic_variables[variable.name].value is None:
         print(" output is None! ")
       else:
         n=n+1
         print(n)
         #print(f"  {variable.name} = {variable.long_name} ({variable.units})")
         f.variables['diag_name'][n]=variable.name
         ejemplo=model.diagnostic_variables[variable.name].value
         list2.append(ejemplo.item())
list2 = [-999 if math.isnan(x) else x for x in list2]
#list1 = [0 if str(x)=='nan' else x for x in list1]
f.variables['diag_value'][:]=list2
#f.variables['diag_name'][:]=list1    

plot *clorofila superficie*, en este caso todo el periodo, el modelo en este punto puede hacer el futuro pero no ha usado observaciones todavía. Plotear las observaciones juntas, hacer alguna métrica?